In [1]:
import json
import os
import sys
from collections import Counter

# --- SETUP DEL PERCORSO ---
# Se il tuo notebook è nella cartella 'test', de-commenta la riga sotto 
# per permettere a Python di "vedere" la cartella superiore 'src' o la root
# sys.path.append('../')

sys.path.append(os.path.abspath('..'))

from send import sending_run  # Assicurati che l'import corrisponda alla tua struttura

# Percorso del file JSON (modificalo se necessario)
TEST_FILE = './test_vectors.json' 
# TEST_FILE = './test_vectors.json' # Usa questo se il notebook è dentro la cartella 'test'

def run_all_sending_tests():
    try:
        with open(TEST_FILE, 'r') as f:
            test_data = json.load(f)
    except FileNotFoundError:
        print(f"❌ Errore: File {TEST_FILE} non trovato. Controlla il percorso!")
        return

    passed = 0
    failed = 0
    failed_tests = []

    print(f"🔍 Trovati {len(test_data)} test nel file.")
    print("🚀 Inizio esecuzione automatica dei test SENDING...")
    print("-" * 70)

    for i, test in enumerate(test_data):
        comment = test.get('comment', f'Test #{i}')
        
        # Controlliamo se esiste la sezione 'sending' per questo test
        if 'sending' not in test or not test['sending']:
            print(f"⏩ Test #{i}: {comment} \n   -> SALTATO (Nessun vettore sending)")
            continue
            
        sending_case = test['sending'][0]
        given = sending_case['given']
        expected = sending_case['expected']

        vin = given['vin']
        recipients = given['recipients']
        
        try:
            # Eseguiamo la tua funzione di invio
            result = sending_run(vin, recipients)
            
            # Estraiamo TUTTI i possibili set di output validi
            expected_outputs_sets = expected.get('outputs', [])
            if not expected_outputs_sets:
                expected_outputs_sets = [[]] # Per gestire correttamente i test senza output previsti
            
            test_passed = False
            # Iteriamo su tutte le combinazioni valide fornite dal JSON
            for expected_outputs in expected_outputs_sets:
                # Confrontiamo le liste ignorando l'ordine
                if Counter(result) == Counter(expected_outputs):
                    test_passed = True
                    break
            
            if test_passed:
                print(f"✅ Test #{i} PASSATO: {comment}")
                passed += 1
            else:
                print(f"❌ Test #{i} FALLITO: {comment}")
                print(f"   Atteso (una di queste combinazioni): {expected_outputs_sets}")
                print(f"   Ottenuto: {result}")
                failed += 1
                failed_tests.append(i)
                
        except Exception as e:
            # Se la funzione genera un'eccezione (es. ValueError per chiavi invalidi)
            expected_outputs_sets = expected.get('outputs', [])
            # Se il test si aspettava effettivamente un fallimento (array vuoto o assente)
            if not expected_outputs_sets or expected_outputs_sets == [[]]:
                print(f"✅ Test #{i} PASSATO: {comment} (Eccezione catturata come previsto: {e})")
                passed += 1
            else:
                print(f"⚠️ Test #{i} ERRORE: {comment}")
                print(f"   Eccezione Python inattesa: {e}")
                failed += 1
                failed_tests.append(i)

    print("-" * 70)
    print("📊 RISULTATI FINALI:")
    print(f"Totale eseguiti: {passed + failed}")
    print(f"🟢 Passati: {passed}")
    print(f"🔴 Falliti: {failed}")
    
    if failed > 0:
        print(f"\n👉 Ti consiglio di controllare i test numero: {failed_tests}")
    else:
        print("\n🎉 CONGRATULAZIONI! Tutti i test Sending sono passati! L'implementazione è perfetta.")

# Lancia il batch!
run_all_sending_tests()

🔍 Trovati 26 test nel file.
🚀 Inizio esecuzione automatica dei test SENDING...
----------------------------------------------------------------------
sender is loading...
selected 2 valid inputs: [{'txid': 'f4184fc596403b9d638783cf57adfe4c75c605f6356fbc91338530e9831e9e16', 'vout': 0, 'scriptSig': '483046022100ad79e6801dd9a8727f342f31c71c4912866f59dc6e7981878e92c5844a0ce929022100fb0d2393e813968648b9753b7e9871d90ab3d815ebf91820d704b19f4ed224d621025a1e61f898173040e20616d43e9f496fba90338a39faa1ed98fcbaeee4dd9be5', 'txinwitness': '', 'prevout': {'scriptPubKey': {'hex': '76a91419c2f3ae0ca3b642bd3e49598b8da89f50c1416188ac'}}, 'private_key': 'eadc78165ff1f8ea94ad7cfdc54990738a4c53f6e0507b42154201b8e5dff3b1'}, {'txid': 'a1075db55d416d3ca199f55b6084e2115b9345e16c5cf302fc80e9d5fbf5d48d', 'vout': 0, 'scriptSig': '48304602210086783ded73e961037e77d49d9deee4edc2b23136e9728d56e4491c80015c3a63022100fda4c0f21ea18de29edbce57f7134d613e044ee150a89e2e64700de2d4e83d4e2103bd85685d03d111699b15d046319febe77f8de